In [4]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64
JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


#### FIX ME #####
# change animal_shelter and AnimalShelter to match your CRUD Python module file name and class name
from CRUD_Python_Module import AnimalShelter


###########################
# Data Manipulation / Model
###########################
# FIX ME update with your username and password and CRUD Python module name

username = "aacuser"
password = "YOUR_PASSWORD_HERE"

# Connect to database via CRUD Module
db = AnimalShelter(username, password)

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents to be returned
df = pd.DataFrame.from_records(db.read({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an
# invalid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will return a new dataframe that does not contain the dropped column(s)
df.drop(columns=['_id'], inplace=True, errors='ignore')

## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)


#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

# FIX ME Add in Grazioso Salvare's logo
image_filename = 'Grazioso Salvare Logo.png'
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

# FIX ME Place the HTML image tag in the line below into the app.layout code according to your design
# FIX ME Also remember to include a unique identifier such as your name or date

app.layout = html.Div([

    # Dashboard branding and logo
    html.Center([
        html.A(
            html.Img(
                src='data:image/png;base64,{}'.format(
                    encoded_image.decode()
                ),
                style={'height': '100px', 'maxWidth': '100%'}
            ),
            href='https://www.snhu.edu',
            target='_blank'
        ),

        html.H1('SNHU CS-340 Dashboard'),
        html.H3('Christina Frediani - CS 340 Project Two')
    ]),

    html.Hr(),

    # Interactive filtering options
    html.Div([
        html.Label("Select Rescue Type:"),

        dcc.RadioItems(
            id='filter-type',
            options=[
                {'label': 'Water Rescue', 'value': 'Water'},
                {'label': 'Mountain or Wilderness Rescue',
                 'value': 'Mountain'},
                {'label': 'Disaster or Individual Tracking',
                 'value': 'Disaster'},
                {'label': 'Reset', 'value': 'Reset'}
            ],
            value='Reset',
            labelStyle={
                'display': 'inline-block',
                'marginRight': '20px'
            }
        )
    ], style={'textAlign': 'center', 'padding': '15px'}),

    html.Hr(),

    # Interactive data table
    dash_table.DataTable(
        id='datatable-id',

        columns=[
            {"name": i, "id": i, "deletable": False, "selectable": True}
            for i in df.columns
        ],

        data=df.to_dict('records'),

        # Set up the features for your interactive data table
        # to make it user-friendly for your client.
        # Module Six DataTable functionality is reused here.

        editable=False,
        filter_action="native",
        sort_action="native",
        sort_mode="multi",
        column_selectable=False,
        row_selectable="single",
        row_deletable=False,
        selected_rows=[0],
        page_action="native",
        page_current=0,
        page_size=10,
        style_table={'overflowX': 'auto'},
        style_cell={
            'textAlign': 'left',
            'minWidth': '100px',
            'maxWidth': '220px',
            'overflow': 'hidden',
            'textOverflow': 'ellipsis'
        }
    ),

    html.Br(),
    html.Hr(),

    # This sets up the dashboard so that your chart
    # and your geolocation chart are side-by-side
    html.Div(
        className='row',
        style={
            'display': 'flex',
            'flexWrap': 'wrap',
            'gap': '20px'
        },
        children=[
            html.Div(
                id='graph-id',
                className='col s12 m6',
                style={'flex': '1 1 450px', 'minWidth': 0}
            ),

            html.Div(
                id='map-id',
                className='col s12 m6',
                style={'flex': '1 1 450px', 'minWidth': 0}
            )
        ]
    )
])


#############################################
# Interaction Between Components / Controller
#############################################

@app.callback(
    Output('datatable-id', 'data'),
    [Input('filter-type', 'value')]
)
def update_dashboard(filter_type):
    ## FIX ME Add code to filter interactive data table with MongoDB queries

    if filter_type == 'Water':
        query = {
            "animal_type": "Dog",
            "breed": {
                "$in": [
                    "Labrador Retriever Mix",
                    "Chesapeake Bay Retriever",
                    "Newfoundland"
                ]
            },
            "sex_upon_outcome": "Intact Female",
            "age_upon_outcome_in_weeks": {
                "$gte": 26,
                "$lte": 156
            }
        }

    elif filter_type == 'Mountain':
        query = {
            "animal_type": "Dog",
            "breed": {
                "$in": [
                    "German Shepherd",
                    "Alaskan Malamute",
                    "Old English Sheepdog",
                    "Siberian Husky",
                    "Rottweiler"
                ]
            },
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {
                "$gte": 26,
                "$lte": 156
            }
        }

    elif filter_type == 'Disaster':
        query = {
            "animal_type": "Dog",
            "breed": {
                "$in": [
                    "Doberman Pinscher",
                    "German Shepherd",
                    "Golden Retriever",
                    "Bloodhound",
                    "Rottweiler"
                ]
            },
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {
                "$gte": 20,
                "$lte": 300
            }
        }

    else:
        # Reset returns the complete unfiltered dataset.
        query = {}

    # Retrieve matching records from MongoDB.
    filtered_df = pd.DataFrame.from_records(db.read(query))

    # Remove MongoDB ObjectId before passing records to Dash.
    filtered_df.drop(
        columns=['_id'],
        inplace=True,
        errors='ignore'
    )

    # Return records in the format required by the DataTable.
    return filtered_df.to_dict('records')


# Display the breeds of animal based on quantity represented in
# the data table
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")]
)
def update_graphs(viewData):
    ###FIX ME ####
    # add code for chart of your choice (e.g. pie chart)

    if viewData is None:
        dff = df.copy()
    else:
        dff = pd.DataFrame.from_records(viewData)

    if dff.empty or 'breed' not in dff.columns:
        return html.Div("No animal records available for this filter.")

    # Count animals by breed.
    breed_counts = (
        dff['breed']
        .fillna('Unknown')
        .value_counts()
    )

    # Show the 10 most common breeds and group the remainder.
    if len(breed_counts) > 10:
        top_breeds = breed_counts.head(10).copy()
        other_count = int(breed_counts.iloc[10:].sum())
        top_breeds['Other Breeds'] = other_count
        breed_counts = top_breeds

    # Prepare the data for Plotly.
    chart_df = breed_counts.reset_index()
    chart_df.columns = ['breed', 'count']

    # Create the pie chart.
    figure = px.pie(
        chart_df,
        names='breed',
        values='count',
        title='Animal Breed Distribution'
    )

    # Improve chart readability.
    figure.update_traces(
        textposition='inside',
        textinfo='percent',
        hovertemplate='%{label}: %{value} animals (%{percent})<extra></extra>'
    )

    figure.update_layout(
        margin=dict(l=20, r=20, t=60, b=20),
        height=500,
        showlegend=True,
        legend=dict(
            orientation='v',
            yanchor='top',
            y=1,
            xanchor='left',
            x=1
        ),
        uniformtext_minsize=10,
        uniformtext_mode='hide'
    )

    return [
        dcc.Graph(
            figure=figure,
            config={'displayModeBar': False},
            style={'width': '100%'}
        )
    ]


# This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    if not selected_columns:
        return []

    return [{
        'if': {'column_id': i},
        'background_color': '#D2F3FF'
    } for i in selected_columns]


# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable

@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")]
)
def update_map(viewData, index):

    # Use the original dataframe until Dash supplies virtual data.
    if viewData is None:
        dff = df.copy()
    else:
        dff = pd.DataFrame.from_records(viewData)

    # Handle empty filtered results.
    if dff.empty:
        return html.Div("No animal locations available.")

    # Because we only allow single row selection, the list can
    # be converted to a row index here.
    if index is None or len(index) == 0:
        row = 0
    else:
        row = index[0]

    # Make sure the selected row exists.
    if row < 0 or row >= len(dff):
        row = 0

    # Use column names so sorting and column order do not affect the map.
    animal = dff.iloc[row]

    latitude = pd.to_numeric(
        animal.get('location_lat'), errors='coerce'
    )
    longitude = pd.to_numeric(
        animal.get('location_long'), errors='coerce'
    )

    if pd.isna(latitude) or pd.isna(longitude):
        return html.Div(
            "Location coordinates are unavailable for this animal."
        )

    breed = str(animal.get('breed') or 'Unknown breed')
    animal_name = str(animal.get('name') or 'Unnamed animal')

    # Austin TX is at [30.75, -97.48]
    return [
        dl.Map(
            style={
                'width': '100%',
                'height': '500px'
            },
            center=[float(latitude), float(longitude)],
            zoom=10,
            children=[
                dl.TileLayer(id="base-layer-id"),

                # Marker with tool tip and popup
                # Column 13 and 14 define the grid-coordinates for the map
                # Column 4 defines the breed for the animal
                # Column 9 defines the name of the animal
                dl.Marker(
                    position=[
                        float(latitude),
                        float(longitude)
                    ],
                    children=[
                        dl.Tooltip(breed),
                        dl.Popup([
                            html.H3("Animal Name"),
                            html.P(animal_name)
                        ])
                    ]
                )
            ]
        )
    ]


# Run app and display result in jupyterlab mode, note, if you have previously run a prior app,
# the default port of 8050 may not be available, if so, try setting an alternate port.
app.run_server()


Dash app running on https://tunnelcamilla-raviolibreak-3000.codio.io/proxy/8050/
